# Bronze Play-by-Play Ingestion

## tl;dr

This notebook wrote three complete source-oriented play-by-play Parquet files for 2023–2025: 147,928 total plays and 58.88 MB on disk.

All files passed round-trip validation with 372 columns, no null or duplicate game/play keys, and all expected WR fields present. The only observed dtype difference was `goal_to_go`: `int32` in 2023 and `float64` in 2024–2025.

No play classifications, team denominators, WR aggregations, or fantasy features are created here.

## Context & Methods

### Scope

- Seasons: 2023–2025
- Environment: `sports_dev_env`
- Source loader: `nflreadpy.load_pbp()`
- In-memory format: pandas after explicit conversion from Polars
- Storage format: one Parquet file per season
- Expected source grain: `game_id + play_id`

This notebook preserves the complete source DataFrame. Selecting plays and defining targets, attempts, red-zone usage, and end-zone usage belong in Silver.

## Setup

In [1]:
from pathlib import Path

import nflreadpy as nfl
import pandas as pd

# Keep the extraction scope explicit and easy to change for future refreshes.
SEASONS = [2023, 2024, 2025]

# Find the repository root whether this runs from Jupyter or nbconvert.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ROADMAP.md").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "ROADMAP.md").exists():
            PROJECT_ROOT = parent
            break
    else:
        raise RuntimeError("Could not locate the repository root.")

PBP_DIR = PROJECT_ROOT / "data/bronze/pbp"
PBP_DIR.mkdir(parents=True, exist_ok=True)

# These fields support the first WR slice or the joins needed to build it.
EXPECTED_WR_COLUMNS = [
    "season",
    "week",
    "season_type",
    "game_id",
    "play_id",
    "posteam",
    "pass_attempt",
    "complete_pass",
    "receiver_player_id",
    "receiver_player_name",
    "air_yards",
    "receiving_yards",
    "yards_gained",
    "yardline_100",
    "goal_to_go",
    "touchdown",
    "pass_touchdown",
]

output_records = []
pbp_columns_by_season = {}
pbp_dtypes_by_season = {}

## Data

## 1. Ingest Play-by-Play by Season

### What we are verifying

For each season:

- The loader returns only the requested season
- `game_id + play_id` is populated and unique
- Regular-season and postseason coverage is visible
- Expected WR and team-opportunity source fields are present
- The complete source DataFrame can be written to Parquet
- The saved file reloads with the same shape and columns

In [2]:
# Process and release one season at a time instead of holding all PBP years in memory.
for season in SEASONS:
    pbp_path = PBP_DIR / f"pbp_{season}.parquet"

    print(f"Loading {season} play-by-play...")
    pbp = nfl.load_pbp(seasons=[season]).to_pandas()

    # Stop before writing if the loader did not honor the requested season.
    assert set(pbp["season"].dropna().unique()) == {season}

    # Check the source grain without filtering or deduplicating Bronze rows.
    pbp_key = ["game_id", "play_id"]
    null_key_rows = pbp[pbp_key].isna().any(axis=1).sum()
    duplicate_key_rows = pbp.duplicated(pbp_key).sum()

    missing_wr_columns = [
        column for column in EXPECTED_WR_COLUMNS if column not in pbp.columns
    ]

    # Save schema details now so cross-season drift can be reviewed later.
    pbp_columns_by_season[season] = pbp.columns.tolist()
    pbp_dtypes_by_season[season] = {
        column: str(dtype) for column, dtype in pbp.dtypes.items()
    }

    season_type_counts = pbp["season_type"].value_counts(dropna=False)

    # Persist every source row and column, then immediately verify the disk copy.
    pbp.to_parquet(pbp_path, index=False)
    pbp_reloaded = pd.read_parquet(pbp_path)

    assert pbp_reloaded.shape == pbp.shape
    assert pbp_reloaded.columns.tolist() == pbp.columns.tolist()

    output_records.append(
        {
            "dataset": "pbp",
            "season": season,
            "path": str(pbp_path),
            "rows": len(pbp),
            "columns": len(pbp.columns),
            "games": pbp["game_id"].nunique(),
            "min_week": pbp["week"].min(),
            "max_week": pbp["week"].max(),
            "regular_season_rows": season_type_counts.get("REG", 0),
            "postseason_rows": season_type_counts.get("POST", 0),
            "null_key_rows": null_key_rows,
            "duplicate_key_rows": duplicate_key_rows,
            "missing_wr_columns": ", ".join(missing_wr_columns),
            "pandas_mb": round(
                pbp.memory_usage(deep=True).sum() / 1024**2,
                2,
            ),
            "file_mb": round(pbp_path.stat().st_size / 1024**2, 2),
            "round_trip_passed": True,
        }
    )

    print(
        f"Saved {pbp_path.name}: "
        f"{len(pbp):,} rows, {len(pbp.columns)} columns"
    )

    # Release both copies before loading the next season.
    del pbp, pbp_reloaded

Loading 2023 play-by-play...


Saved pbp_2023.parquet: 49,665 rows, 372 columns
Loading 2024 play-by-play...


Saved pbp_2024.parquet: 49,492 rows, 372 columns
Loading 2025 play-by-play...


Saved pbp_2025.parquet: 48,771 rows, 372 columns


## Results

## 2. Bronze Output Summary

This table shows what was written and the validation results for each season.

In [3]:
pbp_output_summary = pd.DataFrame(output_records)

display(pbp_output_summary)

print(f"Files written: {len(pbp_output_summary)}")
print(f"Total rows written: {pbp_output_summary['rows'].sum():,}")
print(f"Total Parquet size: {pbp_output_summary['file_mb'].sum():,.2f} MB")
print(
    "All round trips passed:",
    pbp_output_summary["round_trip_passed"].all(),
)

,dataset,season,path,rows,columns,games,min_week,max_week,regular_season_rows,postseason_rows,null_key_rows,duplicate_key_rows,missing_wr_columns,pandas_mb,file_mb,round_trip_passed
0,pbp,2023,/Users/dtwice/Development/dev_sports/nfl/fanta...,49665,372,285,1,22,47399,2266,0,0,,379.22,19.67,True
1,pbp,2024,/Users/dtwice/Development/dev_sports/nfl/fanta...,49492,372,285,1,22,47274,2218,0,0,,377.86,19.73,True
2,pbp,2025,/Users/dtwice/Development/dev_sports/nfl/fanta...,48771,372,285,1,22,46452,2319,0,0,,372.34,19.48,True


Files written: 3
Total rows written: 147,928
Total Parquet size: 58.88 MB
All round trips passed: True


## 3. Cross-Season Schema Check

The 2023 schema is used only as a comparison point. Differences are reported rather than automatically reconciled in Bronze.

In [4]:
reference_season = SEASONS[0]
reference_columns = set(pbp_columns_by_season[reference_season])
reference_dtypes = pbp_dtypes_by_season[reference_season]

schema_records = []

for season in SEASONS:
    season_columns = set(pbp_columns_by_season[season])
    common_columns = reference_columns & season_columns
    dtype_changes = [
        column
        for column in common_columns
        if pbp_dtypes_by_season[season][column]
        != reference_dtypes[column]
    ]

    schema_records.append(
        {
            "season": season,
            "column_count": len(season_columns),
            "missing_vs_2023": sorted(reference_columns - season_columns),
            "extra_vs_2023": sorted(season_columns - reference_columns),
            "dtype_changes_vs_2023": sorted(dtype_changes),
        }
    )

schema_summary = pd.DataFrame(schema_records)
display(schema_summary)

,season,column_count,missing_vs_2023,extra_vs_2023,dtype_changes_vs_2023
0,2023,372,[],[],[]
1,2024,372,[],[],[goal_to_go]
2,2025,372,[],[],[goal_to_go]


## 4. Required WR Column Check

Any missing field would block or change the planned Silver definitions.

In [5]:
wr_column_summary = pbp_output_summary[
    ["season", "columns", "missing_wr_columns"]
].copy()

wr_column_summary["all_expected_wr_columns_present"] = (
    wr_column_summary["missing_wr_columns"] == ""
)

display(wr_column_summary)

,season,columns,missing_wr_columns,all_expected_wr_columns_present
0,2023,372,,True
1,2024,372,,True
2,2025,372,,True


## Takeaways

- Complete play-by-play is stored separately for 2023, 2024, and 2025.
- Processing one season at a time bounds pandas memory usage.
- Candidate key, coverage, WR-column, schema, and Parquet round-trip checks passed for all three files.
- Bronze preserves the observed `goal_to_go` dtype difference; Silver should normalize it deliberately.
- Bronze preserves the complete source and does not define targets, attempts, red-zone usage, or end-zone usage.
- With these files present, the WR-focused Bronze ingestion scope is complete and shared Silver work can begin.